RECUPERATORIO DE IMAGENOLOGIA MEDICA

MODULARIZACION

FUNCION PRINCIPAL, por ejemplo recu.py, o pruebas.py


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label

# importamos TODO lo de nuestros módulos
from pruebas.carga import cargar_imagen_con_barra
from pruebas.histogramas import mostrar_hist_y_cdf, ecualizar_por_cdf, mediciones_por_labels
from pruebas.mascaras import (
    suavizar_mediana,
    crear_mascara_umbral,
    cerrar_mascara,
    etiquetar_y_overlay,
    extraer_objeto_por_indice,
)
from pruebas.filtros import aplicar_filtros_suavizado
from pruebas.bordes import detectar_bordes_sobel_y_kernel

# carpeta de salida (si quieres guardar luego)
OUTDIR = os.path.join("pruebas", "output")
os.makedirs(OUTDIR, exist_ok=True)

# ruta de la imagen principal (la de MR o la que estés usando en clase)
RUTA_IMAGEN = "Data/SCD2001_006/SCD2001_MR_117.dcm"
# también puedes poner: RUTA_IMAGEN = "imagenes/hand1.jpg"

def mostrar_imagen(im, titulo="Imagen"):
    plt.figure(figsize=(5,5))
    plt.imshow(im, cmap='gray')
    plt.title(titulo)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def main():
    # =========================
    # 2.1 Cargar imagen
    # =========================
    im = cargar_imagen_con_barra(RUTA_IMAGEN, vmin=0, vmax=160, tick_interval=20)
    print("Forma:", im.shape)
    print("Tipo de dato:", im.dtype)
    # si es DICOM la función ya mostró metadatos
    mostrar_imagen(im, "Imagen original")

    # =========================
    # 2.2 Histograma + CDF + ecualización
    # =========================
    hist, cdf = mostrar_hist_y_cdf(im)
    im_eq = ecualizar_por_cdf(im, cdf)   # esta es nuestra ecualizada
    mostrar_imagen(im_eq, "Imagen ecualizada")

    # =========================
    # 2.3 Máscara por umbral + cierre + etiquetas
    # (esto es la parte tipo ventrículo del apunte)
    # =========================
    im_filt = suavizar_mediana(im, size=3)           # median_filter
    mask_ini = crear_mascara_umbral(im_filt, 60)     # np.where(im_filt>60,1,0)
    mask_cerrada = cerrar_mascara(mask_ini, 1)       # binary_closing
    labels, nlabels, overlay = etiquetar_y_overlay(mask_cerrada)
    print("Número de objetos etiquetados:", nlabels)

    # mediciones como en el ejemplo (media, varianza por label)
    mediciones_por_labels(im, labels, indices=[1, 5])

    # intentar extraer el objeto con label 5 (como en tu guía)
    im_obj = extraer_objeto_por_indice(im, labels, index=5)

    # =========================
    # 2.4 Filtros (los de la mano)
    # =========================
    # esto lee otra imagen ('imagenes/hand1.jpg') dentro del módulo
    aplicar_filtros_suavizado("imagenes/hand1.jpg")

    # =========================
    # 2.5 Bordes (Sobel + kernels) sobre la misma de la mano
    # =========================
    bordes_kernel, bordes_sobel = detectar_bordes_sobel_y_kernel("imagenes/hand1.jpg")

    # =========================
    # Extra: calcular área, media y varianza del objeto más grande
    # a partir de la máscara cerrada (igual que el final de tu ejemplo)
    # =========================
    labels_mask, nlabels2 = label(mask_cerrada)
    if nlabels2 > 0:
        areas = [np.sum(labels_mask == (i+1)) for i in range(nlabels2)]
        idx_max = int(np.argmax(areas)) + 1
        pixeles_obj = im[labels_mask == idx_max]
        mean_obj = pixeles_obj.mean()
        var_obj = pixeles_obj.var()
        print(f"Objeto mayor (label {idx_max}): media={mean_obj:.2f}, varianza={var_obj:.2f}")
    else:
        print("No se encontraron objetos para medir.")

if __name__ == "__main__":
    main()


__init__.py

In [ ]:
# Este archivo marca el directorio como un paquete Python.
# No necesita contenido obligatorio, pero puedes importar todo aquí
# para facilitar el uso desde pruebas.py si lo deseas.

from .carga import cargar_imagen_con_barra
from .histogramas import mostrar_hist_y_cdf, ecualizar_por_cdf, mediciones_por_labels
from .mascaras import (
    suavizar_mediana,
    crear_mascara_umbral,
    cerrar_mascara,
    etiquetar_y_overlay,
    extraer_objeto_por_indice,
)
from .filtros import aplicar_filtros_suavizado
from .bordes import detectar_bordes_sobel_y_kernel

__all__ = [
    "cargar_imagen_con_barra",
    "mostrar_hist_y_cdf",
    "ecualizar_por_cdf",
    "mediciones_por_labels",
    "suavizar_mediana",
    "crear_mascara_umbral",
    "cerrar_mascara",
    "etiquetar_y_overlay",
    "extraer_objeto_por_indice",
    "aplicar_filtros_suavizado",
    "detectar_bordes_sobel_y_kernel",
]


carga.py


In [ ]:
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

def cargar_imagen_con_barra(ruta, vmin=0, vmax=255, tick_interval=50):
    """
    Carga y muestra una imagen DICOM o JPG con barra de color y ticks.
    """
    im = imageio.imread(ruta)
    print(f"Data type: {im.dtype}")
    print(f"Shape: {im.shape}")

    if hasattr(im, "meta"):
        print("Metadatos disponibles:", im.meta.keys())

    tick_vals = np.arange(vmin, vmax + tick_interval, tick_interval)

    plt.imshow(im, vmin=vmin, vmax=vmax, cmap="gray")
    cbar = plt.colorbar()
    cbar.set_ticks(tick_vals)
    plt.title("Imagen cargada")
    plt.tight_layout()
    plt.show()

    return im


histogramas.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndi

def mostrar_hist_y_cdf(im):
    """
    Calcula histograma y CDF usando ndi.histogram, y los grafica.
    """
    hist = ndi.histogram(im, min=0, max=255, bins=256)
    cdf = hist.cumsum() / hist.sum()

    fig, axes = plt.subplots(2, 1, sharex=True)
    axes[0].plot(hist, label="Histograma")
    axes[1].plot(cdf, label="CDF")
    for ax in axes:
        ax.legend(loc="center right")
    plt.tight_layout()
    plt.show()
    return hist, cdf

def ecualizar_por_cdf(im, cdf):
    """
    Ecualiza la imagen usando la CDF normalizada (como en los ejemplos de clase).
    """
    im_eq = (cdf[im] * 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(im, cmap="gray")
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(im_eq, cmap="gray")
    axes[1].set_title("Ecualizada")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
    return im_eq

def mediciones_por_labels(im, labels, indices=[1, 5]):
    """
    Calcula medias, varianzas y gráficos de histograma por etiqueta.
    """
    print("Media total:", ndi.mean(im))
    print("Media etiquetas:", ndi.mean(im, labels))
    print(f"Media etiquetas {indices}:", ndi.mean(im, labels, index=indices))

    var_total = ndi.variance(im)
    print("Varianza total:", var_total)
    var_sel = ndi.variance(im, labels, index=indices)
    print(f"Varianza etiquetas {indices}:", var_sel)

    obj_hists = ndi.histogram(im, 0, 255, 256, labels, index=indices)
    for i, idx in enumerate(indices):
        plt.plot(obj_hists[i], label=f"Label {idx}")
    plt.legend()
    plt.show()

mascaras.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndi

def suavizar_mediana(im, size=3):
    """
    Aplica un filtro de mediana para suavizar intensidades (ruido).
    """
    im_filt = ndi.median_filter(im, size=size)
    return im_filt

def crear_mascara_umbral(im_filt, umbral=60):
    """
    Crea una máscara binaria con valores 1 donde im_filt > umbral.
    """
    mask = np.where(im_filt > umbral, 1, 0)
    return mask

def cerrar_mascara(mask, iterations=1):
    """
    Aplica operación de cierre morfológico para rellenar huecos.
    """
    mask_closed = ndi.binary_closing(mask, iterations=iterations)
    return mask_closed

def etiquetar_y_overlay(mask):
    """
    Etiqueta regiones conectadas en la máscara y crea overlay tipo rainbow.
    """
    labels, nlabels = ndi.label(mask)
    print(f"Número de etiquetas: {nlabels}")
    overlay = np.where(labels != 0, labels, np.nan)

    plt.imshow(overlay, cmap="rainbow", alpha=0.75)
    plt.title("Overlay de etiquetas")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    return labels, nlabels, overlay

def extraer_objeto_por_indice(im, labels, index=1):
    """
    Extrae y muestra el objeto especificado por su índice de etiqueta.
    """
    mask_obj = np.where(labels == index, 1, 0).astype(int)
    bboxes = ndi.find_objects(mask_obj)

    if not bboxes or bboxes[0] is None:
        print(f"No se encontró el objeto con etiqueta {index}")
        return None

    print("Índice del bounding box:", bboxes[0])
    im_crop = im[bboxes[0]]

    plt.imshow(im_crop, cmap="gray")
    plt.title(f"Objeto recortado (label {index})")
    plt.axis("off")
    plt.show()
    return im_crop


filtros.py

In [ ]:
import imageio.v2 as imageio
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndi

def _leer_gris(ruta):
    im = imageio.imread(ruta)
    if im.ndim == 3:
        im = np.mean(im, axis=2).astype(np.uint8)
    else:
        im = im.astype(np.uint8)
    return im

def aplicar_filtros_suavizado(ruta_imagen):
    """
    Aplica filtros de media, mediana y gaussiano sobre una imagen.
    """
    im = _leer_gris(ruta_imagen)

    weights = np.ones((3,3)) * (1/9)
    im_media = ndi.convolve(im, weights)
    im_mediana = ndi.median_filter(im, size=5)
    im_gauss = ndi.gaussian_filter(im, sigma=1.5)

    fig, axes = plt.subplots(1,4, figsize=(12,3))
    axes[0].imshow(im, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(im_media, cmap='gray'); axes[1].set_title('Media'); axes[1].axis('off')
    axes[2].imshow(im_mediana, cmap='gray'); axes[2].set_title('Mediana'); axes[2].axis('off')
    axes[3].imshow(im_gauss, cmap='gray'); axes[3].set_title('Gauss'); axes[3].axis('off')
    plt.tight_layout()
    plt.show()
    return im_media, im_mediana, im_gauss


bordes.py

In [ ]:
import imageio.v2 as imageio
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndi

def detectar_bordes_sobel_y_kernel(ruta_imagen):
    """
    Detecta bordes con Sobel y kernels personalizados (horizontal/vertical).
    """
    im = imageio.imread(ruta_imagen)
    if im.ndim == 3:
        im = np.mean(im, axis=2).astype(np.uint8)
    else:
        im = im.astype(np.uint8)

    # Calcular histograma y CDF para ecualización
    hist = ndi.histogram(im, min=0, max=255, bins=256)
    cdf = hist.cumsum() / hist.sum()
    im_eq = (cdf[im] * 255).astype(np.uint8)

    # Máscara simple (brillo alto)
    mask_bone = im_eq >= 196

    # ---- Kernels ----
    Kx = np.array([[-1,-1,-1],[0,0,0],[1,1,1]])
    Ky = np.array([[-1,0,1],[-1,0,1],[-1,0,1]])

    Ix = ndi.convolve(mask_bone.astype(float), Kx)
    Iy = ndi.convolve(mask_bone.astype(float), Ky)
    bordes_kernel = np.sqrt(Ix**2 + Iy**2)

    # ---- Sobel ----
    sobel_x = ndi.sobel(mask_bone, axis=0)
    sobel_y = ndi.sobel(mask_bone, axis=1)
    bordes_sobel = np.sqrt(sobel_x**2 + sobel_y**2)

    fig, axes = plt.subplots(2,3, figsize=(10,6))
    axes[0,0].imshow(mask_bone, cmap='gray'); axes[0,0].set_title('Máscara')
    axes[0,1].imshow(Ix, cmap='gray'); axes[0,1].set_title('Kx')
    axes[0,2].imshow(Iy, cmap='gray'); axes[0,2].set_title('Ky')
    axes[1,0].imshow(bordes_kernel, cmap='gray'); axes[1,0].set_title('Bordes Kernel')
    axes[1,1].imshow(bordes_sobel, cmap='gray'); axes[1,1].set_title('Sobel Magnitud')
    axes[1,2].axis('off')
    for ax in axes.ravel(): ax.axis('off')
    plt.tight_layout()
    plt.show()

    return bordes_kernel, bordes_sobel
